# Compare stream recommendation models

This notebook consumes the larger live-stream snapshot from `00_get_data.ipynb` and compares a staged recommendation pipeline:
1. SQL-style audience eligibility filtering
2. lightweight popularity and diversity reduction
3. derived features
4. a model-ready preprocessing and regression pipeline
5. top-50 inference

The complex model currently learns from a transparent heuristic proxy because interaction labels are not available yet. Replace that proxy with clicks, watch time, follows, or skips when those labels are collected.

## Load the live-stream snapshot

Run `00_get_data.ipynb` first. The default collection is now up to 5,000 streams, controlled by `TWITCH_LIVE_STREAM_PAGES`. Twitch's available live catalog varies continuously, so this is a sampling window rather than a complete count of every live stream.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

notebook_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
data_path = notebook_dir / 'data' / 'live_stream_features.jsonl'
if not data_path.exists():
    raise FileNotFoundError(f'Run 00_get_data.ipynb first: {data_path}')

streams_df = pd.read_json(data_path, lines=True)
print(f'Loaded {len(streams_df)} streams from {data_path}')

Loaded 5000 streams from /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/live_stream_features.jsonl


## Derive features

These features are deliberately available before model inference. Follower and subscriber counts are optional because Twitch does not expose them in the public `/streams` response; they can be joined later from authorized channel metadata.

In [2]:
derived_df = streams_df.copy()
numeric_columns = ['viewer_count', 'viewer_log1p', 'stream_age_hours', 'tag_count', 'title_length', 'follower_count', 'subscriber_count']
for column in numeric_columns:
    if column in derived_df:
        derived_df[column] = pd.to_numeric(derived_df[column], errors='coerce')

derived_df['title_word_count'] = derived_df['title'].fillna('').str.split().str.len()
derived_df['stream_freshness'] = np.exp(-derived_df['stream_age_hours'].fillna(0).clip(lower=0) / 6)
derived_df['viewer_percentile'] = derived_df['viewer_count'].rank(pct=True)
derived_df['category_viewer_median'] = derived_df.groupby('game_name')['viewer_count'].transform('median')
derived_df['category_viewer_median_log1p'] = np.log1p(derived_df['category_viewer_median'].fillna(0))
derived_df['has_audience_metadata'] = derived_df[['follower_count', 'subscriber_count']].notna().any(axis=1) if {'follower_count', 'subscriber_count'}.issubset(derived_df.columns) else False

print('Available audience metadata:', [column for column in ['follower_count', 'subscriber_count'] if column in derived_df.columns])
derived_df.head(3)

Available audience metadata: []


,id,user_id,user_login,user_name,game_id,game_name,title,language,tags,is_mature,...,tag_count,title_length,is_followed_channel,matches_recent_vod_category,title_word_count,stream_freshness,viewer_percentile,category_viewer_median,category_viewer_median_log1p,has_audience_metadata
0,318188581220,92038375,caedrel,Caedrel,21779,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,en,"[English, LPL, LCK, LEC, Worlds, Esports, Drop...",False,...,7,35,False,False,6,0.502424,1.0000,260.0,5.56452,False
1,318190806372,27115917,kamet0,Kamet0,21779,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,fr,"[Français, LEC, leagueoflegends, esport, RLCS,...",False,...,10,108,False,False,19,0.874666,0.9998,260.0,5.56452,False
2,318190614756,83232866,ibai,ibai,21779,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,es,"[Español, IRL, Chatting, KOI, DropsActivados]",False,...,5,112,False,False,25,0.835018,0.9996,260.0,5.56452,False


## Stage 1: SQL-style eligibility filter

This mirrors a first-pass database query. The viewer threshold is enforceable from the Twitch stream payload. Follower and subscriber thresholds are only applied when those columns exist and contain data; otherwise the notebook reports that they were unavailable instead of treating missing values as zero.

In [3]:
MIN_VIEWERS = 1000
MIN_FOLLOWERS = 1000
MIN_SUBSCRIBERS = 100
REQUIRE_AUDIENCE_METADATA = False

sql_candidates_df = derived_df.query('viewer_count >= @MIN_VIEWERS').copy()
filter_results = {'loaded': len(derived_df), 'viewer_threshold': len(sql_candidates_df)}

for column, threshold in [('follower_count', MIN_FOLLOWERS), ('subscriber_count', MIN_SUBSCRIBERS)]:
    if column not in sql_candidates_df or not sql_candidates_df[column].notna().any():
        print(f'Skipping {column} threshold: Twitch stream data does not include usable {column}.')
        continue
    if REQUIRE_AUDIENCE_METADATA:
        sql_candidates_df = sql_candidates_df[sql_candidates_df[column].notna()]
    sql_candidates_df = sql_candidates_df[sql_candidates_df[column].fillna(0) >= threshold]
    filter_results[column] = len(sql_candidates_df)

print(filter_results)

Skipping follower_count threshold: Twitch stream data does not include usable follower_count.
Skipping subscriber_count threshold: Twitch stream data does not include usable subscriber_count.
{'loaded': 5000, 'viewer_threshold': 403}


## Stage 2: lightweight preliminary models

Popularity keeps the strongest audience candidates. The diversity pass limits each category so the expensive model sees multiple kinds of streams instead of only one dominant game.

In [4]:
PRELIMINARY_LIMIT = 1500
MAX_STREAMS_PER_CATEGORY = 100

popularity_candidates_df = sql_candidates_df.sort_values(
    ['viewer_count', 'viewer_log1p'], ascending=False
).head(PRELIMINARY_LIMIT).copy()

diverse_candidates_df = (
    popularity_candidates_df
    .sort_values('viewer_count', ascending=False)
    .groupby('game_name', dropna=False, sort=False)
    .head(MAX_STREAMS_PER_CATEGORY)
    .reset_index(drop=True)
)

print({
    'after_sql_filter': len(sql_candidates_df),
    'after_popularity_model': len(popularity_candidates_df),
    'after_diversity_model': len(diverse_candidates_df),
    'categories': diverse_candidates_df['game_name'].nunique(),
})

{'after_sql_filter': 403, 'after_popularity_model': 403, 'after_diversity_model': 403, 'categories': 130}


## Stage 3: complex model pipeline

The preprocessing pipeline imputes numeric values, scales them, one-hot encodes category and language, and fits a gradient-boosted regression model. Until behavioral labels exist, `proxy_target` is a weighted heuristic used only to prove the pipeline and create an initial ranking.

In [5]:
model_df = diverse_candidates_df.copy()
model_numeric = [
    'viewer_log1p', 'stream_age_hours', 'tag_count', 'title_length',
    'title_word_count', 'stream_freshness', 'viewer_percentile',
    'category_viewer_median_log1p', 'is_followed_channel',
    'matches_recent_vod_category',
]
model_categorical = ['game_name', 'language']
model_numeric = [column for column in model_numeric if column in model_df.columns]
model_categorical = [column for column in model_categorical if column in model_df.columns]

for column in model_numeric:
    model_df[column] = pd.to_numeric(model_df[column], errors='coerce')
for column in model_categorical:
    model_df[column] = model_df[column].fillna('unknown').astype(str)

proxy_weights = {
    'viewer_log1p': 0.45,
    'stream_freshness': 0.10,
    'viewer_percentile': 0.15,
    'is_followed_channel': 0.15,
    'matches_recent_vod_category': 0.10,
    'category_viewer_median_log1p': 0.05,
}
proxy_target = pd.Series(0.0, index=model_df.index)
for column, weight in proxy_weights.items():
    if column in model_df:
        values = model_df[column].fillna(0).astype(float)
        spread = values.max() - values.min()
        proxy_target += weight * ((values - values.min()) / spread if spread else 0)

numeric_pipeline = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipeline = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, model_numeric), ('categorical', categorical_pipeline, model_categorical)], remainder='drop')
complex_model = Pipeline([('features', preprocessor), ('regressor', GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=3, random_state=42, loss='huber'))])
complex_model.fit(model_df[model_numeric + model_categorical], proxy_target)
model_df['complex_score'] = complex_model.predict(model_df[model_numeric + model_categorical])
model_df = model_df.sort_values(['complex_score', 'viewer_count'], ascending=False).reset_index(drop=True)
model_df.insert(0, 'rank', model_df.index + 1)
model_df[['rank', 'user_name', 'game_name', 'title', 'viewer_count', 'complex_score']].head(50)

,rank,user_name,game_name,title,viewer_count,complex_score
0,1,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,66175,0.571629
1,2,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,36421,0.571629
2,3,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,109134,0.570420
3,4,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,28350,0.550001
4,5,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,137490,0.543878
5,6,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],27898,0.535959
6,7,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31062,0.534673
7,8,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",26387,0.530429
8,9,ohnePixel,Just Chatting,🔴JAPAN STORY TIME🔴,22956,0.508772
9,10,deepins02,Just Chatting,[ти таник] | !winline !aliexpress !tg,18323,0.507451


In [6]:
top_50_complex_df = model_df.head(50).copy()
output_path = notebook_dir / 'data' / 'complex_model_top_50.jsonl'
pipeline_path = notebook_dir / 'data' / 'pipeline_candidates.jsonl'
top_50_complex_df.to_json(output_path, orient='records', lines=True, date_format='iso')
model_df.to_json(pipeline_path, orient='records', lines=True, date_format='iso')
print(f'Wrote {len(top_50_complex_df)} final recommendations to {output_path}')
print(f'Wrote {len(model_df)} model candidates to {pipeline_path}')

Wrote 50 final recommendations to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/complex_model_top_50.jsonl
Wrote 403 model candidates to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/pipeline_candidates.jsonl
